# Causal Conv3D

In this notebook, we’ll understand one of the core building blocks used in modern video generation architectures:

- Causal Conv3D
- Temporal Padding
- Dilation
- Receptive Fields
- Why causality matters in video generation



---


Ok so let’s start with the **Causal Convolution 3D layer**.

If you remember, in Transformers we have a **causal attention mask**, which prevents the model from seeing future tokens during autoregressive generation.

For example:

- token `t5` can only attend to previous tokens
- it cannot see `t6`, `t7`, ...

Now in video generation, we do something very similar.

Instead of generating words, we generate frames over time.

So you can think of a video like:

```text
frame₀ → frame₁ → frame₂ → ...
```

which is somewhat similar to:

```text
word₀ → word₁ → word₂ → ...
```

---

# 🎥 Why Do We Need Causal Conv3D?

In video models, we usually work with tensors shaped like:

```text
(T × H × W)
```

where:

- `T` = time / frames
- `H` = height
- `W` = width

For simplicity, let’s assume:

```text
Video Shape = 8 × 32 × 32
```

meaning:

- 8 frames
- each frame is 32×32

---

Now suppose we use a normal `Conv3D` layer with kernel size:

```text
3 × 3 × 3
```

A standard Conv3D is **not causal**.

So when processing frame `t`, it may look at:

```text
[t-1, t, t+1]
```

But during autoregressive video generation, this is a problem.

Why?

Because frame `t+1` is a future frame.

The model should not have access to future information while generating the current frame.

---

# Causal Conv3D

So what Causal Conv3D does is, it takes

```text
[t-2, t-1, t]
```

So the model only sees the past and present frames.

---

# Now at t = 0?

At the first frame:

```text
t = 0
```

there are no previous frames.

So we pad the beginning of the video along the temporal dimension.

Think of it like adding black frames at the start.

---

# Temporal Padding

So instead of:

```text
[t0, t1, t2, t3, ...]
```

we transform it into:

```text
[pad, pad, t0, t1, t2, ...]
```

Now when the convolution operates at `t0`, it sees:

```text
[pad, pad, t0]
```


# 🛠️ PyTorch Implementation

The key ideas are:

- Left-only temporal padding
- Symmetric spatial padding
- Use of dilation


In [ ]:
# vae/conv.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalConv3d(nn.Module):
    """
    Conv3d that is causal in the time dimension.
    Spatial dims (H, W) use standard symmetric padding.
    Time dim uses left-only padding so the kernel never sees future frames.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int | tuple,
        stride: int | tuple = 1,
        dilation: int | tuple = 1,
        bias: bool = True,
    ):
        super().__init__()

        # Normalize to tuples: (T, H, W)
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size, kernel_size)
        if isinstance(stride, int):
            stride = (stride, stride, stride)
        if isinstance(dilation, int):
            dilation = (dilation, dilation, dilation)

        self.kernel_size = kernel_size
        self.stride = stride
        self.dilation = dilation

        # How much to pad on each side for each dim
        # For spatial: symmetric → pad = (k-1)//2 each side
        # For time:    causal   → pad = (k-1)*d on left, 0 on right
        kt, kh, kw = kernel_size
        dt, dh, dw = dilation

        self.time_pad = (kt - 1) * dt   # left-pad only

        pad_h = (kh - 1) * dh // 2
        pad_w = (kw - 1) * dw // 2

        # We handle time padding manually in forward();
        # pass spatial padding directly to Conv3d
        self.conv = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            dilation=dilation,
            padding=(0, pad_h, pad_w),   # time=0, we do it manually
            bias=bias,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, T, H, W)
        if self.time_pad > 0:
            # F.pad pads last dims first, so pad = (W_left, W_right, H_left, H_right, T_left, T_right)
            x = F.pad(x, (0, 0, 0, 0, self.time_pad, 0))
        return self.conv(x)


class CausalConv3d_1x1(nn.Module):
    """Pointwise (1×1×1) conv — no causal padding needed, but keeps the API consistent."""

    def __init__(self, in_channels: int, out_channels: int, bias: bool = True):
        super().__init__()
        self.conv = nn.Conv3d(in_channels, out_channels, kernel_size=1, bias=bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)

# Common Questions

## Whats Dilation ?

Normally when we say kernel size of `3`, we take:

```text
[n-1, n, n+1]
```

But with dilation, we introduce gaps between the sampled positions.

For example:

```text
dilation = 2
```

would make the kernel look like:

```text
[n-2, n, n+2]
```

instead of:

```text
[n-1, n, n+1]
```

---

## Why Use Dilation?

Dilation increases the **receptive field** of the convolution without increasing the kernel size.

That means:

- the model can see a larger context
- without adding many more parameters
- and without increasing computation too much

This becomes very useful in video models because motion and temporal patterns can span across multiple frames.

For example:

- small dilation → local motion
- larger dilation → longer-range temporal understanding

---

## Why Is Temporal Padding Calculated Like This?

In the implementation, we use:

```python
pad_h = (kh - 1) * dh // 2 
pad_w = (kw - 1) * dw // 2
time_pad = (kt - 1) * dt
```

---

This padding is done so that:

1. the convolution remains causal
2. future frames are never visible
3. the output size does not shrink too quickly

---

Without padding, every convolution would reduce to very small dimensions.

For example:

```text
Input : 8x32x32
Kernel size : 3
dilation : 2
```

The effective kernel size becomes:

```text
effective kernel size
= (kernel - 1) * dilation + 1

= (3 - 1) * 2 + 1
= 5
```

So the convolution is effectively using a:

```text
5x5x5
```

receptive field.

Without padding:

```text
Output size formula:

output =
(input - effective_kernel) + 1
```

Temporal dimension:

```text
T = (8 - 5) + 1 = 4
```

Height:

```text
H = (32 - 5) + 1 = 28
```

Width:

```text
W = (32 - 5) + 1 = 28
```

So the output becomes:

```text
4x28x28
```

which is much smaller than the original input.

After multiple layers, the video size would shrink very quickly.

---

With padding:

```python
pad_h = (5 - 1) // 2 = 2
pad_w = (5 - 1) // 2 = 2
time_pad = 4
```

we preserve the dimensions much better.

The output remains:

```text
8x32x32
```

while still maintaining causality.

---

## Why Left Padding Only?

In normal Conv3D, padding is usually symmetric:

```text
[left_pad, right_pad]
```

But in causal convolution, right padding would expose future frames.

So we only pad on the LEFT side:

```text
[pad, pad, t0, t1, t2, ...]
```

This preserves:

-temporal size  
-causality  
-autoregressive behavior

---

## Spatial Padding vs Temporal Padding

Notice that:

- Height/Width use symmetric padding
- Time uses left-only padding

Why?

Because images do not have a "future" spatial direction.

For spatial dimensions:

```text
(H, W)
```

seeing neighboring pixels is completely fine.

But for time:

```text
(T)
```

future frames must remain hidden.

So:

| Dimension | Padding Type |
|---|---|
| Time | Left-only causal padding |
| Height | Symmetric padding |
| Width | Symmetric padding |

---

# Key Takeaways

- Causal Conv3D prevents future frame leakage
- It behaves similarly to causal attention in Transformers
- Temporal padding is applied only on the LEFT side
- Spatial dimensions use normal symmetric padding
- Dilation increases the receptive field without increasing kernel size
- Proper padding helps preserve output dimensions across deeper networks
